# 5 3 content analysis he prior nmr

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import get_source_corrections

source_corrections = get_source_corrections(window_days=28, horizon_days=28).query('prior_corrections==0 and nmr_count_28d>1').reset_index(drop=True)
print(source_corrections.shape)
source_corrections.head()

In [ ]:
from utils import StataDiD, he_content_factors, prepare_content_count, get_coefs

means_yes, std_errs_yes, stats_yes, pvals_yes, clls_yes, chls_yes = [], [], [], [], [], []
means_no, std_errs_no, stats_no, pvals_no, clls_no, chls_no = [], [], [], [], [], []
means_ratio, std_errs_ratio, stats_ratio, pvals_ratio, clls_ratio, chls_ratio = [], [], [], [], [], []

for item in he_content_factors:
    fvar, fvar_name, user_var, filename, tcol, external_threshold = item[0], item[1], item[2], item[3], item[4], item[5]
    print('------------------------------')
    print(fvar_name)
    if fvar == 'misleadingness':
        filename = 'original_posts_wide_prior_nmr.parquet.gzip'
        tcol = 'weeks_from_display'

    post_counts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
    post_counts_by_var = prepare_content_count(filename, fvar, tcol, external_threshold)
    post_counts = post_counts.merge(post_counts_by_var, on=['tweet_id', tcol], how='left')
    post_counts[f'tweet_count_{fvar}_yes'] = post_counts[f'tweet_count_{fvar}_yes'].fillna(0)
    post_counts[f'tweet_count_{fvar}_no'] = post_counts[f'tweet_count_{fvar}_no'].fillna(0)
    post_counts[f'{fvar}_ratio'] = post_counts[f'tweet_count_{fvar}_yes']/post_counts['tweet_count']
    post_counts = post_counts.merge(source_corrections, on='tweet_id', how='inner')
    stata_did = StataDiD(post_counts, 'treated', 'tweet_id', tcol, -4, 4)

    print('[Group 1]', end=' ')
    stata_did.tw_did(reg=f'qui ppmlhdfe', dev=f'tweet_count_{fvar}_yes', random_unit=False, cluster='account_cluster')
    mean_yes, std_err_yes, stat_yes, pval_yes, cll_yes, chl_yes = get_coefs(f'qui ppmlhdfe', 2, 3, stat_type='z', irr_or=1, unit_group=True, full_return=True)

    print('[Group 0]', end=' ')
    stata_did.tw_did(reg=f'qui ppmlhdfe', dev=f'tweet_count_{fvar}_no', random_unit=False, cluster='account_cluster')
    mean_no, std_err_no, stat_no, pval_no, cll_no, chl_no = get_coefs(f'qui ppmlhdfe', 2, 3, stat_type='z', irr_or=1, unit_group=True, full_return=True)

    print('Ratio', end=' ')
    stata_did.tw_did(reg=f'qui xtreg', dev=f'{fvar}_ratio', random_unit=False, cluster='account_cluster', condition=f'if tweet_count>0 & ({fvar}_ratio>=0 & {fvar}_ratio<=1)')
    mean_ratio, std_err_ratio, stat_ratio, pval_ratio, cll_ratio, chl_ratio = get_coefs(f'qui xtreg', 2, 3, stat_type='t', unit_group=True, full_return=True)

    means_yes.extend(mean_yes)
    std_errs_yes.extend(std_err_yes)
    stats_yes.extend(stat_yes)
    pvals_yes.extend(pval_yes)
    clls_yes.extend(cll_yes)
    chls_yes.extend(chl_yes)

    means_no.extend(mean_no)
    std_errs_no.extend(std_err_no)
    stats_no.extend(stat_no)
    pvals_no.extend(pval_no)
    clls_no.extend(cll_no)
    chls_no.extend(chl_no)

    means_ratio.append(mean_ratio)
    std_errs_ratio.append(std_err_ratio)
    stats_ratio.append(stat_ratio)
    pvals_ratio.append(pval_ratio)
    clls_ratio.append(cll_ratio)
    chls_ratio.append(chl_ratio)

yes_label = 'Yes'
no_label = 'No'
ratio_label = 'Ratio'

hes_yes = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [yes_label]*len(he_content_factors),
                        'est': means_yes, 'std_err': std_errs_yes, 'stat': stats_yes, 'pval': pvals_yes,
                        'cll': clls_yes, 'chl': chls_yes})
hes_no = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [no_label]*len(he_content_factors),
                        'est': means_no, 'std_err': std_errs_no, 'stat': stats_no, 'pval': pvals_no,
                        'cll': clls_no, 'chl': chls_no})

hes_groups = pd.concat([hes_yes, hes_no], axis=0, ignore_index=True)

hes_ratio = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [ratio_label]*len(he_content_factors),
                        'est': means_ratio, 'std_err': std_errs_ratio, 'stat': stats_ratio, 'pval': pvals_ratio,
                        'cll': clls_ratio, 'chl': chls_ratio})